# DCPA on NVIDIA DGX Spark（GB10 Grace Blackwell · 128 GB 统一内存）一键无脑训练工作台

本 Notebook 专为 **NVIDIA DGX Spark**（ARM64 `aarch64` + GB10 Blackwell GPU + 128 GB LPDDR5X 统一内存 + 本地高速 NVMe）定制，**无需 Google Drive、无 Colab 12 小时断连限制、打开后从 Cell 1 顺序执行到 Cell 8 即可完成全量下载 → 校验 → 10 万步底座训练 → 8,000 步全能对话/工具微调 → 导出至 Mac M3 Pro**。

---

## 第 0 步：首次在 DGX Spark 上启动本 Notebook（仅需做一次，二选一）

先把本代码仓库传到 DGX Spark（`git clone` 或 `scp`），在 DGX Spark 终端进入仓库根目录后执行：

### 方案 A（强烈推荐：NVIDIA 官方 NGC PyTorch 容器，原生内置 GB10 Blackwell CUDA/Triton/bf16）
```bash
bash scripts/setup_dgx_spark.sh --ngc
```

### 方案 B（直接在 DGX OS 宿主机建独立 Python venv，自动装 ARM64 `cu128` PyTorch）
```bash
bash scripts/setup_dgx_spark.sh --native
```

### 在 Mac 浏览器打开 JupyterLab（SSH 端口转发）
```bash
ssh -L 8888:localhost:8888 <你的用户名>@<DGX_Spark_IP>
# 然后在 Mac 浏览器打开 http://localhost:8888 -> 点击 notebooks/DGX_Spark_DCPA_Train.ipynb
```

---

## 本 Notebook 会自动流式下载并构建哪些数据？（总计约 `730 MB`，全部 `streaming=True` 不爆盘）

| 序号 | 落盘目录 | 目标体积 | 数据源与作用 |
|---|---|---|---|
| 1 | `artifacts/data/` | `~450 MB` | 五域底座语料（EN FineWeb-Edu / ZH FineWeb-Edu-Chinese-V2.1 / JA Wiki+C4 / Python & Rust `tree-sitter` 零错误 AST 代码） |
| 2 | `artifacts/cap_data/cap_reasoning_*` | `~100 MB` | 紧凑符号推理（`< 240B` 草稿）+ `40%` 潜空间隐式蒸馏（OpenMathInstruct-2/MetaMathQA/GSM8K + ARC-Challenge + CRUXEval + 4~9变量逻辑状态机） |
| 3 | `artifacts/cap_data/cap_writing_*` | `~100 MB` | 反八股深度写作 + 前置 `< 180B` 提纲锁槽（Slot-0 Pinning）+ `style_id` (`1=memo/2=expository/3=narrative`) |
| 4 | `artifacts/cap_data/cap_toolcall_*` | `~60 MB` | 海马体单行工具签名 + `30%` OOV 生造别名 + `60%` 链式 / `20%` 报错自愈 / `20%` 零调用克制（SmolTalk `apigen-80k` + Hermes + 可选 xLAM） |
| 5 | `artifacts/sft_data/` | `~20 MB` | Stage 5 四桶语用对话微调语料（指令问答 + 多轮指代 + 零废话代码 + 拒答边界） |


In [ ]:
# Cell 1 [全局配置开关 —— 只需按需修改这里，下面所有单元格直接无脑顺序运行]
import os
import sys
from pathlib import Path

# ==============================================================================
# 1. 模型规模开关（DGX Spark 拥有 128 GB 统一内存，推荐直接用 "210m"！）
#    "210m" -> 208.65M 总参 / 22.86M 单步激活 (32微专家 Top-2, D=768, L_max=8)
#    "48m"  ->  47.62M 总参 / 15.77M 单步激活 ( 8微专家 Top-2, D=512, L_max=8)
# ==============================================================================
MODEL_PRESET = "210m"

# ==============================================================================
# 2. 快速全链路冒烟模式开关：
#    True  -> 每库仅下载/生成 2 MB，底座跑 200 步 + Stage 5 跑 50 步（约 2 分钟验证整机无坑）
#    False -> 正式全量：下载 730 MB 全量语料 + 底座 100,000 步 + Stage 5 8,000 步
# ==============================================================================
SMOKE_TEST_ONLY = False

# ==============================================================================
# 3. 可选高级项（一般保持默认即可）
# ==============================================================================
COMPILE_MODE = "auto"         # "auto" | "reduce-overhead" | "none"（若极早期驱动 Triton 报错可改为 "none"）
FORCE_REBUILD_DATA = False    # True = 强制删除旧数据重新流式拉取；False = 已有分片自动秒跳过
HF_TOKEN = os.environ.get("HF_TOKEN", "")        # 可选：填 HuggingFace Token 可额外补充 gated xLAM
HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "")  # 可选：国内加速镜像例如 "https://hf-mirror.com"

# 自动定位仓库根目录（无论在根目录还是 notebooks/ 子目录启动 JupyterLab 都能精准找到）
cwd = Path.cwd().resolve()
if (cwd / "train" / "train_dcpa.py").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "train" / "train_dcpa.py").exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"未找到 DCPA 仓库根目录，当前目录: {cwd}")

os.chdir(REPO_ROOT)
for _p in (str(REPO_ROOT), str(REPO_ROOT / "dcpa-train-py")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

os.environ["PYTHONPATH"] = f"{REPO_ROOT}:{REPO_ROOT / 'dcpa-train-py'}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["DCPA_MODEL_PRESET"] = MODEL_PRESET
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if HF_ENDPOINT:
    os.environ["HF_ENDPOINT"] = HF_ENDPOINT

if MODEL_PRESET == "210m":
    CONFIG_YAML = str(REPO_ROOT / "configs" / "dcpa_210m_moe.yaml")
    PRESET_NAME = "moe_210m"
elif MODEL_PRESET == "48m":
    CONFIG_YAML = str(REPO_ROOT / "configs" / "dcpa_48m_train.yaml")
    PRESET_NAME = "proto_48m"
else:
    raise ValueError(f"MODEL_PRESET 必须为 '48m' 或 '210m'，当前: {MODEL_PRESET!r}")

DATA_DIR = REPO_ROOT / "artifacts" / "data"
CAP_DATA_DIR = REPO_ROOT / "artifacts" / "cap_data"
SFT_DATA_DIR = REPO_ROOT / "artifacts" / "sft_data"
CKPT_DIR = REPO_ROOT / "artifacts" / "checkpoints"
LOGS_DIR = REPO_ROOT / "artifacts" / "logs"
EXPORT_DIR = REPO_ROOT / "artifacts" / "export"
for _d in (DATA_DIR, CAP_DATA_DIR, SFT_DATA_DIR, CKPT_DIR, LOGS_DIR, EXPORT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"[Config] REPO_ROOT       = {REPO_ROOT}")
print(f"[Config] MODEL_PRESET    = {MODEL_PRESET} ({PRESET_NAME} -> {Path(CONFIG_YAML).name})")
print(f"[Config] SMOKE_TEST_ONLY = {SMOKE_TEST_ONLY} | COMPILE_MODE = {COMPILE_MODE}")
print(f"[Config] HF_ENDPOINT     = {HF_ENDPOINT or '(official huggingface.co)'} | HF_TOKEN={'SET' if HF_TOKEN else 'not set (ungated sources OK)'}")


In [ ]:
# Cell 2 [DGX Spark 硬件/CUDA 体检 + 全套依赖一键安装]
import os
import platform
import shutil
import subprocess
import sys

print(f"[Host] Arch={platform.machine()} | OS={platform.platform()} | Python={sys.version.split()[0]}")
if hasattr(os, "sysconf"):
    try:
        ram_gb = (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")) / (1024 ** 3)
        print(f"[Host] Unified/System RAM = {ram_gb:.1f} GB")
    except Exception:
        pass
disk = shutil.disk_usage(REPO_ROOT)
print(f"[Disk] Free NVMe space at {REPO_ROOT}: {disk.free / (1024 ** 3):.1f} GB")

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
if smi.returncode == 0:
    print("\n".join(smi.stdout.splitlines()[:14]))
else:
    print("[WARN] nvidia-smi 未返回 GPU 信息，请确认已用 --gpus all 启动容器或在 DGX OS 主机运行！")

# 1. 检查 PyTorch + CUDA（若未装 CUDA 版则自动拉取 ARM64 cu128 轮子）
need_torch_install = False
try:
    import torch
    if not torch.cuda.is_available():
        print("[WARN] 当前 PyTorch 未启用 CUDA，尝试安装 CUDA 12.8 (Blackwell) 适配版 PyTorch...")
        need_torch_install = True
except ImportError:
    need_torch_install = True

if need_torch_install:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", "torch", "torchvision",
         "--index-url", "https://download.pytorch.org/whl/cu128"],
        check=True,
    )
    import importlib
    import torch
    importlib.reload(torch)

# 2. 一键安装全部数据流式下载、tree-sitter AST 过滤与可视化依赖
deps = [
    "safetensors",
    "datasets",
    "hf_transfer",
    "tree-sitter",
    "tree-sitter-python",
    "tree-sitter-rust",
    "pyyaml",
    "pytest",
    "matplotlib",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(REPO_ROOT / "dcpa-train-py")],
    check=True,
)

# 3. 打印 DCPA 跨平台设备层最终识别结果与模型参数量
from dcpa.model import DCPAModel
from train.train_dcpa import build_dcpa_config, load_yaml_config
from utils.device import describe_hardware_profile, detect_hardware_profile

profile = detect_hardware_profile("auto")
print("\n" + describe_hardware_profile(profile))
if profile.device_type != "cuda":
    raise RuntimeError(
        "PyTorch 仍未识别到 DGX Spark GPU！请先用官方 NGC 容器启动：bash scripts/setup_dgx_spark.sh --ngc"
    )

cfg_preview = build_dcpa_config(PRESET_NAME, load_yaml_config(CONFIG_YAML))
m_counts = DCPAModel(cfg_preview).count_parameters()
print(
    f"[Model Check] {PRESET_NAME}: total={m_counts['total']:,} ({m_counts['total']/1e6:.2f}M) | "
    f"active/step={m_counts['active_per_step']:,} ({m_counts['active_per_step']/1e6:.2f}M) | "
    f"E={cfg_preview.num_experts} top_k={cfg_preview.top_k_experts} L_max={cfg_preview.l_max}"
)


In [ ]:
# Cell 3 [全量 730 MB 纯字节语料一键流式下载与构建（已有分片自动跳过）]
import json
import subprocess
import sys
import time
from pathlib import Path

t_all = time.perf_counter()
smoke_flag = ["--smoke-quota"] if SMOKE_TEST_ONLY else []

def _run_builder(label: str, sentinel: Path, cmd: list[str]) -> None:
    if sentinel.exists() and sentinel.stat().st_size > 1024 and not FORCE_REBUILD_DATA:
        mb = sentinel.stat().st_size / (1024 ** 2)
        print(f"[SKIP][{label}] 已存在本地 NVMe 分片 {sentinel.name} ({mb:.2f} MB)，跳过重复下载。")
        return
    print(f"\n[BUILD][{label}] Running: {' '.join(cmd)}", flush=True)
    t0 = time.perf_counter()
    subprocess.run(cmd, check=True)
    mb = sentinel.stat().st_size / (1024 ** 2) if sentinel.exists() else 0.0
    print(f"[DONE][{label}] {sentinel.name} = {mb:.2f} MB ({time.perf_counter() - t0:.1f}s)", flush=True)

# 1) ~450 MB 五域底座语料（EN/ZH/JA/Python/Rust，含 tree-sitter AST 零错误过滤）
_run_builder(
    "Base Corpus (450MB)",
    DATA_DIR / "train.bin",
    [sys.executable, "data/build_corpus.py", "--out-dir", str(DATA_DIR), *smoke_flag],
)

# 2) ~100 MB 紧凑逻辑推理 + 40% 潜空间内隐蒸馏语料
_run_builder(
    "Cap Reasoning (100MB)",
    CAP_DATA_DIR / "cap_reasoning_train.bin",
    [sys.executable, "data/build_cap_reasoning.py", "--out-dir", str(CAP_DATA_DIR), "--target-mb", "100", *smoke_flag],
)

# 3) ~100 MB 反八股长文 + 逆向提纲锁槽 + 三语体 style_id 写作语料
_run_builder(
    "Cap Writing (100MB)",
    CAP_DATA_DIR / "cap_writing_train.bin",
    [sys.executable, "data/build_cap_writing.py", "--out-dir", str(CAP_DATA_DIR), "--target-mb", "100", *smoke_flag],
)

# 4) ~60 MB 海马体单行工具签名 + 30% OOV 生造别名 + 错误自愈工具语料
_run_builder(
    "Cap ToolCall (60MB)",
    CAP_DATA_DIR / "cap_toolcall_train.bin",
    [sys.executable, "data/build_cap_toolcall.py", "--out-dir", str(CAP_DATA_DIR), "--target-mb", "60", *smoke_flag],
)

# 5) ~20 MB Stage 5 四桶语用对话微调语料
_run_builder(
    "Stage 5 SFT (20MB)",
    SFT_DATA_DIR / "sft_train.bin",
    [sys.executable, "data/build_sft_stage5.py", "--out-dir", str(SFT_DATA_DIR), *smoke_flag],
)

# 汇总所有数据分片体积
print("\n" + "=" * 72)
print(f"[Data Ready] 全部语料准备就绪（总耗时 {time.perf_counter() - t_all:.1f}s）：")
total_mb = 0.0
for p in sorted([*DATA_DIR.glob("*.bin"), *CAP_DATA_DIR.glob("*.bin"), *SFT_DATA_DIR.glob("*.bin")]):
    if p.name.endswith(".loss_mask.bin"):
        continue
    mb = p.stat().st_size / (1024 ** 2)
    total_mb += mb
    print(f"  - {str(p.relative_to(REPO_ROOT)):44s} : {mb:7.2f} MB")
print(f"  >>> 纯字节文本总量合计: {total_mb:.2f} MB")
print("=" * 72)


In [ ]:
# Cell 4 [DGX Spark GB10 上线全套复检：因果律 / STE / 数据纯净度 / 五大能力 / GPU 吞吐基准]
import os
import subprocess
import sys
import time

t0 = time.perf_counter()
env = dict(os.environ)
env["DCPA_DATA_DIR"] = str(DATA_DIR)

test_files = [
    "tests/test_sanity.py",
    "tests/test_data_pipeline.py",
    "tests/test_capabilities_upgrade.py",
    "tests/test_perf_and_compile.py",
]
subprocess.run([sys.executable, "-m", "pytest", "-v", "-s", *test_files], env=env, check=True)
print(f"\n[DGX Spark Preflight] 全套 4 组硬核验收 + GPU 吞吐基准 100% GREEN（耗时 {time.perf_counter() - t0:.1f}s）!")


In [ ]:
# Cell 5 [一键启动 / 断电自动续跑 100,000 步底座训练（Stage 1 -> Stage 4）]
# ==============================================================================
# 提示：若希望关掉电脑通宵后台跑，可直接复制下方打印出的 [Train Command] 到 DGX Spark 的 tmux 中运行：
#   tmux new -s dcpa
#   <粘贴命令回车>
#   按 Ctrl+B 然后按 D 后台挂起；随时 tmux attach -t dcpa 查看进度！
# ==============================================================================
import os
import subprocess
import sys

total_steps = 200 if SMOKE_TEST_ONLY else 100000
save_every = 50 if SMOKE_TEST_ONLY else 1000
probe_every = 50 if SMOKE_TEST_ONLY else 500
stage_bounds_args = ["--stage-boundaries", "0,50,120,160,200"] if SMOKE_TEST_ONLY else []

train_cmd = [
    sys.executable,
    "-u",
    "train/train_dcpa.py",
    "--config", CONFIG_YAML,
    "--preset", PRESET_NAME,
    "--total-steps", str(total_steps),
    *stage_bounds_args,
    "--seq-len", "2048",
    "--data-dir", str(DATA_DIR),
    "--cap-data-dir", str(CAP_DATA_DIR),
    "--ckpt-dir", str(CKPT_DIR),
    "--logs-dir", str(LOGS_DIR),
    "--export-dir", str(EXPORT_DIR),
    "--local-cache-dir", str(REPO_ROOT / "artifacts" / "nvme_cache"),
    "--save-every-steps", str(save_every),
    "--save-every-minutes", "10",
    "--probe-every-steps", str(probe_every),
    "--heartbeat-seconds", "60",
    "--resume", "auto",
    "--compile-mode", COMPILE_MODE,
    "--max-consecutive-meltdowns", "3",
]

print("[Train Command] " + " ".join(train_cmd), flush=True)
log_file = LOGS_DIR / "spark_stage1_4_stdout.log"
with log_file.open("a", encoding="utf-8") as lf:
    proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        lf.write(line)
    exit_code = int(proc.wait())

if hasattr(os, "sync"):
    os.sync()
base_sf = EXPORT_DIR / "dcpa_final.safetensors"
if exit_code == 0 and base_sf.exists():
    print(f"\n[Stage 1-4 DONE] 底座训练完工！导出权重: {base_sf} ({base_sf.stat().st_size / (1024**2):.2f} MB)")
elif exit_code == 42:
    raise RuntimeError("[CIRCUIT_BREAKER] 连续 3 次看门狗回滚仍未恢复，请检查 artifacts/logs/watchdog_events.jsonl")
else:
    raise RuntimeError(f"底座训练异常退出，exit_code={exit_code}")


In [ ]:
# Cell 6 [一键启动 Stage 5：8,000 步语用对话 + 推理 / 文风写作 / 海马体工具全能微调]
import os
import subprocess
import sys

BASE_WEIGHTS = EXPORT_DIR / "dcpa_final.safetensors"
CHAT_WEIGHTS = EXPORT_DIR / "dcpa_chat_sft.safetensors"
if not BASE_WEIGHTS.exists():
    raise FileNotFoundError(f"未找到底座权重 {BASE_WEIGHTS}，请先运行 Cell 5 完成 Stage 1~4 训练！")

sft_steps = 50 if SMOKE_TEST_ONLY else 8000
sft_cmd = [
    sys.executable,
    "-u",
    "train/train_stage5_sft.py",
    "--weights", str(BASE_WEIGHTS),
    "--data-dir", str(SFT_DATA_DIR),
    "--cap-data-dir", str(CAP_DATA_DIR),
    "--export-dir", str(EXPORT_DIR),
    "--total-steps", str(sft_steps),
    "--target-tokens-bytes-per-step", "65536",
    "--max-doc-bytes", "2048",
    "--compile-mode", COMPILE_MODE,
    "--log-every", "25" if SMOKE_TEST_ONLY else "50",
]

print("[Stage 5 Command] " + " ".join(sft_cmd), flush=True)
subprocess.run(sft_cmd, check=True)
if hasattr(os, "sync"):
    os.sync()
print(f"\n[Stage 5 DONE] 最终对话与工具权重已导出: {CHAT_WEIGHTS} ({CHAT_WEIGHTS.stat().st_size / (1024**2):.2f} MB)")


In [ ]:
# Cell 7 [训练曲线可视化 + 最新视觉探针报告 + 一键传回 Mac M3 Pro 命令]
import json
import math
import socket

metrics_path = LOGS_DIR / "metrics.jsonl"
if metrics_path.exists():
    rows = [json.loads(line) for line in metrics_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"[Telemetry] 共读取 {len(rows)} 条训练记录 ({metrics_path})")
    if rows:
        steps = [r.get("step", i + 1) for i, r in enumerate(rows)]
        losses = [r.get("loss_total", 0.0) for r in rows]
        bpcs = [r.get("loss_lang", 0.0) / math.log(2.0) for r in rows]
        rhos = [r.get("compression_ratio", 0.0) for r in rows]
        try:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            axes[0].plot(steps, losses); axes[0].set_title("Total Loss"); axes[0].set_xlabel("Step")
            axes[1].plot(steps, bpcs, color="tab:orange"); axes[1].set_title("Language BPC"); axes[1].set_xlabel("Step")
            axes[2].plot(steps, rhos, color="tab:green"); axes[2].set_title("SGSC Compression rho"); axes[2].set_xlabel("Step")
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            print(f"[Plot Skipped] {exc} | Latest step={steps[-1]}, loss={losses[-1]:.4f}, bpc={bpcs[-1]:.3f}, rho={rhos[-1]:.2f}x")
else:
    print("[Telemetry] 尚未生成 metrics.jsonl（运行 Cell 5 后自动产出）。")

probe_md = LOGS_DIR / "latest_probe.md"
if probe_md.exists():
    print("\n" + "-" * 72 + "\n[Latest Visual Probe]\n" + "-" * 72)
    print(probe_md.read_text(encoding="utf-8")[:3000])

chat_sf = EXPORT_DIR / "dcpa_chat_sft.safetensors"
cfg_js = EXPORT_DIR / "config.json"
host = socket.gethostname()
print("\n" + "=" * 72)
print("[回传 Mac M3 Pro 本地极速交互终端命令]")
print(f"1. 在 Mac 终端拷贝权重 (约 {chat_sf.stat().st_size / (1024**2):.1f} MB):" if chat_sf.exists() else "1. 训完后在 Mac 终端拷贝权重:")
print(f"   scp <user>@{host}:{EXPORT_DIR}/dcpa_chat_sft.safetensors <user>@{host}:{EXPORT_DIR}/config.json ./artifacts/export/")
print("2. 在 Mac M3 Pro 启动本地 O(1) 对话 + 安全 AST 工具终端:")
print("   python cli/chat_m3.py --weights artifacts/export/dcpa_chat_sft.safetensors --device mps")
print("=" * 72)


In [ ]:
# Cell 8 [在 DGX Spark 上直接加载刚训完的权重试聊 + 实测海马体工具沙箱]
from cli.chat_m3 import DCPAChatSession, execute_tool_call_safe
from train.train_stage5_sft import load_dcpa_model_from_weights
from utils.device import get_device

weights_to_test = EXPORT_DIR / "dcpa_chat_sft.safetensors"
if not weights_to_test.exists():
    weights_to_test = EXPORT_DIR / "dcpa_final.safetensors"
if not weights_to_test.exists():
    raise FileNotFoundError("尚未找到导出权重，请先跑完 Cell 5 / Cell 6！")

dev = get_device("auto")
chat_model = load_dcpa_model_from_weights(weights_to_test, device=dev)
session = DCPAChatSession(chat_model, device=dev, temp=0.3, top_p=0.9, max_new_bytes=192)
print(f"[Loaded] {weights_to_test.name} on {dev} | 预置海马体工具数: {len(session.tool_signatures)}")

# 验证安全 AST 工具沙箱（calc / now_hk / 恶意拦截）
print("[Sandbox calc]  ", execute_tool_call_safe('calc("(128 * 3 + 16) / 4")', session.tool_registry))
print("[Sandbox now_hk]", execute_tool_call_safe('now_hk()', session.tool_registry))
print("[Sandbox block] ", execute_tool_call_safe("__import__('os').system('id')", session.tool_registry))

# 切换文风并试跑 2 轮 O(1) 有状态推理
session.handle_command("/style memo")
for prompt in (
    "记住新概念：‘星曜缓存’是一种固定8槽的零KV内存索引。请用一句话总结它的优势。",
    "刚才提到的星曜缓存有多少个固定槽位？",
):
    reply, tel = session.chat_turn(prompt)
    print(f"\nyou > {prompt}")
    print(f"dcpa> {reply}")
    print(f"      {tel.format_line()}")
